# Last-Mile Logistics Network Optimization
## Vehicle Routing & Optimal Facility Placement · Chicago, IL

[![Python](https://img.shields.io/badge/Python-3.11-blue)](https://www.python.org)
[![OR-Tools](https://img.shields.io/badge/OR--Tools-9.8-green)](https://developers.google.com/optimization)
[![OSMnx](https://img.shields.io/badge/OSMnx-1.9-orange)](https://osmnx.readthedocs.io)
[![PuLP](https://img.shields.io/badge/PuLP-2.7-yellow)](https://coin-or.github.io/pulp)

A spatial optimization project that solves the **Capacitated Vehicle Routing Problem (CVRP)** and **p-median facility location** on Chicago's real road network.

**Skills Demonstrated:** OSMnx · GeoPandas · NetworkX · Linear Programming · OR-Tools · R-Tree Spatial Indexing · Folium

---

### Project Phases
| Phase | Name | Description |
|-------|------|-------------|
| 1 | Setup & Data Acquisition | Download Chicago road network; create demand points |
| 2 | Distance Matrix & Spatial Indexing | Haversine + network distances; R-tree index |
| 3 | Facility Location (p-Median) | LP solver to find optimal warehouse sites |
| 4 | VRP Solver | Route vehicles with OR-Tools CVRP |
| 5 | Distance Metric Comparison | Haversine vs. network; detour factor analysis |
| 6 | Interactive Folium Dashboard | Full interactive HTML map |
| 7 | Results Summary | Key metrics and portfolio packaging guide |

---
## Setup

### Option A -- Local (conda)

From the repo root:

```bash
conda env create -f environment.yml
conda activate logistics_optimization
jupyter lab notebooks/last_mile_logistics_optimization.ipynb
```

Run all cells top to bottom. Phase 1 downloads Chicago's road network from OpenStreetMap (30-60s), and Phase 6 snaps the vehicle routes onto the road graph (30-90s) -- both need an internet connection on first run.

### Option B -- Google Colab

Run the Colab installation cell below -- it installs all dependencies via pip automatically. No local environment needed.

### Troubleshooting

| Error | Fix |
|-------|-----|
| OSMnx download fails | Check internet; try a smaller place string (e.g. `'Lincoln Park, Chicago'`); wait a few minutes and retry |
| PuLP returns `None` | Check for NaN in the distance matrix; try increasing `p` |
| OR-Tools finds no solution | Increase `time_limit_s`; raise vehicle capacity or `NUM_VEHICLES` |
| NetworkX path not found | Re-run Phase 1 with `retain_all=False` to keep only the largest connected component |
| Folium map blank | Hard-refresh (Ctrl+Shift+R) or try a different browser |
| Memory error on distance matrix | Reduce the number of demand points; use `dtype=np.float32` |

Full setup reference: [`SETUP.md`](../SETUP.md).


## Environment Setup (Google Colab Only)
**Skip this cell if running locally with conda.** See `SETUP.md` for local environment instructions.

In [ ]:
# ── Colab Installation (skip if using local conda environment) ────────────
# Run this cell first in Google Colab — takes ~3 minutes
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install osmnx==1.9.3 geopandas folium ortools==9.8.3296 pulp==2.7.0 rtree -q
    print('Colab packages installed!')
else:
    print('Local environment detected — skipping pip install.')

---
## Phase 1 — Data Acquisition
Download Chicago's real road network from OpenStreetMap and create synthetic delivery demand points.

> **Concept:** OpenStreetMap (OSM) is a free, community-maintained world map. OSMnx downloads any city's road network as a NetworkX graph with a single line of code.

In [ ]:
# ── Step 1: Import all libraries ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Geospatial
import osmnx as ox
import geopandas as gpd
import networkx as nx
import folium
from shapely.geometry import Point
from rtree import index as rtree_index
from math import radians, cos, sin, asin, sqrt

# Optimization
import pulp
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# Settings
np.random.seed(42)
ox.settings.log_console = False
CRS_GEOGRAPHIC = 'EPSG:4326'   # WGS84 lat/lon
CRS_PROJECTED  = 'EPSG:32616'  # UTM Zone 16N (meters, best for Chicago)

print('All imports successful.')

In [ ]:
# ── Step 2: Download Chicago drive network ────────────────────────────────
# network_type='drive' → only roads usable by cars
# Takes 30–60 seconds on first run
print('Downloading Chicago road network...')
G = ox.graph_from_place(
    'Chicago, Illinois, USA',
    network_type='drive',
    retain_all=False,   # Keep only the largest connected component
    simplify=True       # Simplify geometry for faster computation
)

nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

# Reproject to UTM (meters) for accurate distance calculations
G_proj = ox.project_graph(G, to_crs=CRS_PROJECTED)
nodes_proj, edges_proj = ox.graph_to_gdfs(G_proj)

print(f'Network downloaded!')
print(f'  Nodes (intersections): {len(nodes_gdf):,}')
print(f'  Edges (road segments):  {len(edges_gdf):,}')
print(f'  Total road length:      {edges_proj["length"].sum()/1000:.0f} km')

In [ ]:
# ── Step 3: Visualize the road network ────────────────────────────────────
fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_color='#1B3A5C',
    edge_linewidth=0.4,
    bgcolor='white',
    figsize=(12, 10),
    show=False,
    close=False
)
ax.set_title('Chicago Drive Network — OpenStreetMap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/chicago_network.png', dpi=150, bbox_inches='tight')
plt.show()
print('Network plot saved → outputs/chicago_network.png')

In [ ]:
# ── Step 4: Generate 150 synthetic customer demand points ─────────────────
# Points are sampled from actual road-network nodes (guarantees road access)
all_nodes = list(G.nodes())
customer_node_ids = np.random.choice(all_nodes, size=150, replace=False)

customer_coords  = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in customer_node_ids]
customer_lats    = [c[0] for c in customer_coords]
customer_lons    = [c[1] for c in customer_coords]
customer_demand  = np.random.randint(1, 10, size=150)

customers_gdf = gpd.GeoDataFrame({
    'node_id': customer_node_ids,
    'demand':  customer_demand,
    'geometry': [Point(lon, lat) for lat, lon in customer_coords]
}, crs=CRS_GEOGRAPHIC)

print(f'Created {len(customers_gdf)} customer demand points')
print(f'  Total demand:           {customers_gdf["demand"].sum()} packages')
print(f'  Average demand/stop:    {customers_gdf["demand"].mean():.1f} packages')

In [ ]:
# ── Step 5: Define 20 candidate warehouse sites ────────────────────────────
warehouse_node_ids = np.random.choice(all_nodes, size=20, replace=False)
warehouse_coords   = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in warehouse_node_ids]
warehouse_lats     = [c[0] for c in warehouse_coords]
warehouse_lons     = [c[1] for c in warehouse_coords]

warehouses_gdf = gpd.GeoDataFrame({
    'node_id': warehouse_node_ids,
    'site_id': [f'WH_{i:02d}' for i in range(20)],
    'geometry': [Point(lon, lat) for lat, lon in warehouse_coords]
}, crs=CRS_GEOGRAPHIC)

print(f'Created {len(warehouses_gdf)} candidate warehouse sites')
print(warehouses_gdf[['site_id', 'geometry']].head())

---
## Phase 2 — Distance Matrix & Spatial Indexing
Build two distance matrices: fast Haversine (straight-line) and real road network distances.

> **Concept:** A distance matrix is a 2D array where `[i][j]` = distance from location i to location j.  
> An **R-tree** spatial index makes proximity queries O(log n) instead of O(n).

In [ ]:
# ── Step 1: Build R-tree spatial index on candidate warehouse sites ────────
customers_proj  = customers_gdf.to_crs(CRS_PROJECTED)
warehouses_proj = warehouses_gdf.to_crs(CRS_PROJECTED)

warehouse_idx = rtree_index.Index()
for i, row in warehouses_proj.iterrows():
    x, y = row.geometry.x, row.geometry.y
    warehouse_idx.insert(i, (x, y, x, y))  # insert(id, (minx,miny,maxx,maxy))

print('R-tree spatial index built on warehouse sites')

# Example: find the 3 nearest warehouses to the first customer
cx, cy = customers_proj.iloc[0].geometry.x, customers_proj.iloc[0].geometry.y
nearest_3 = list(warehouse_idx.nearest((cx, cy, cx, cy), num_results=3))
print(f'3 nearest warehouses to customer 0: site indices {nearest_3}')

In [ ]:
# ── Step 2: Haversine distance matrix — VECTORIZED with NumPy ─────────────
# Instead of 3,000 sequential Python loop iterations, we broadcast the
# computation across the full matrix in one shot (~100x faster). This
# scales to thousands of locations without code changes.
def haversine_distance(lat1, lon1, lat2, lon2):
    """Scalar great-circle distance between two GPS coordinates (meters)."""
    R = 6_371_000
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi    = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1) * cos(phi2) * sin(dlambda/2)**2
    return 2 * R * asin(sqrt(a))

def haversine_matrix_vectorized(lats1, lons1, lats2, lons2):
    """
    Vectorized Haversine: returns matrix of shape (len(lats1), len(lats2)).
    Uses NumPy broadcasting — no Python-level loops.
    """
    R = 6_371_000
    lat1 = np.radians(np.asarray(lats1))[:, np.newaxis]  # column vector
    lon1 = np.radians(np.asarray(lons1))[:, np.newaxis]
    lat2 = np.radians(np.asarray(lats2))[np.newaxis, :]  # row vector
    lon2 = np.radians(np.asarray(lons2))[np.newaxis, :]
    dphi    = lat2 - lat1
    dlambda = lon2 - lon1
    a = np.sin(dphi/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

n_warehouses = len(warehouses_gdf)
n_customers  = len(customers_gdf)

haversine_matrix = haversine_matrix_vectorized(
    warehouse_lats, warehouse_lons,
    customer_lats,  customer_lons
)

print(f'Haversine distance matrix shape: {haversine_matrix.shape}')
print(f'  Min:  {haversine_matrix.min()/1000:.2f} km')
print(f'  Max:  {haversine_matrix.max()/1000:.2f} km')
print(f'  Mean: {haversine_matrix.mean()/1000:.2f} km')

In [ ]:
# ── Step 3: Network distance helper (real road routing) ────────────────────
# NOTE: Computing all 20×150 = 3,000 pairs via Dijkstra takes several minutes.
# We demonstrate the function here; the full matrix is used in Phase 5.
def network_distance_from_origin(G, origin_lat, origin_lon, dest_coords):
    """
    Compute road network distances from one origin to a list of (lat, lon) destinations.
    Returns a list of distances in meters.
    """
    origin_node = ox.distance.nearest_nodes(G, X=origin_lon, Y=origin_lat)
    dest_lons   = [d[1] for d in dest_coords]
    dest_lats   = [d[0] for d in dest_coords]
    dest_nodes  = ox.distance.nearest_nodes(G, X=dest_lons, Y=dest_lats)
    lengths     = nx.single_source_dijkstra_path_length(G, origin_node, weight='length')
    return [lengths.get(dn, float('inf')) for dn in dest_nodes]

# Test: compute network distances from warehouse 0 to all customers
wh0 = warehouses_gdf.iloc[0]
customer_coord_list = list(zip(customer_lats, customer_lons))
print('Computing network distances from warehouse 0...')
net_distances_wh0 = network_distance_from_origin(
    G, wh0.geometry.y, wh0.geometry.x, customer_coord_list
)
print(f'Computed {len(net_distances_wh0)} network distances')
print(f'  Min: {min(net_distances_wh0)/1000:.2f} km')
print(f'  Max: {max(net_distances_wh0)/1000:.2f} km')

---
## Phase 3 — Facility Location (p-Median Solver)
Given 20 candidate warehouse sites and 150 customers, find the **best 3 warehouses** to open to minimize total weighted distance.

> **Concept:** The **p-median problem** chooses p locations from candidates to minimize Σ(demand × distance) from each customer to its nearest facility. Solved with Integer Linear Programming via PuLP.

In [ ]:
# ── Step 1: Solve the p-Median Linear Program ──────────────────────────────
def solve_p_median(dist_matrix, demand, p):
    """
    Solve the p-median facility location problem via ILP.
    dist_matrix: shape (n_facilities, n_customers)
    demand:      array of customer demand weights
    p:           number of facilities to open
    Returns: (selected_facility_indices, objective_value)
    """
    n_fac  = dist_matrix.shape[0]
    n_cust = dist_matrix.shape[1]

    prob = pulp.LpProblem('p_median', pulp.LpMinimize)

    # y[j] = 1 if warehouse j is opened
    y = [pulp.LpVariable(f'y_{j}', cat='Binary') for j in range(n_fac)]
    # x[i][j] = 1 if customer i is assigned to warehouse j
    x = [[pulp.LpVariable(f'x_{i}_{j}', cat='Binary')
          for j in range(n_fac)] for i in range(n_cust)]

    # Objective: minimize total weighted distance
    prob += pulp.lpSum(
        demand[i] * dist_matrix[j][i] * x[i][j]
        for i in range(n_cust) for j in range(n_fac)
    )

    # Constraint 1: open exactly p facilities
    prob += pulp.lpSum(y) == p
    # Constraint 2: each customer assigned to exactly one facility
    for i in range(n_cust):
        prob += pulp.lpSum(x[i][j] for j in range(n_fac)) == 1
    # Constraint 3: customers assigned only to open facilities
    for i in range(n_cust):
        for j in range(n_fac):
            prob += x[i][j] <= y[j]

    solver = pulp.PULP_CBC_CMD(msg=False)
    prob.solve(solver)

    selected = [j for j in range(n_fac) if pulp.value(y[j]) > 0.5]
    return selected, pulp.value(prob.objective)


# ── Run for p = 3, 4, 5 warehouses ────────────────────────────────────────
print('Solving p-median for p = 3, 4, 5 warehouses...')
results = {}
for p in [3, 4, 5]:
    selected_idx, obj = solve_p_median(haversine_matrix, customer_demand, p)
    results[p] = {'sites': selected_idx, 'objective': obj}
    print(f'  p={p}: sites {selected_idx}, total weighted dist = {obj/1e6:.2f}M m')

OPTIMAL_WAREHOUSES = results[3]['sites']
print(f'\nSelected warehouses for p=3: {OPTIMAL_WAREHOUSES}')

In [ ]:
# ── Step 2: Plot the coverage improvement (elbow) curve ────────────────────
p_values   = list(range(1, 7))
objectives = []
for p in p_values:
    _, obj = solve_p_median(haversine_matrix, customer_demand, p)
    objectives.append(obj / 1e6)

improvements = [objectives[0]] + [
    objectives[i-1] - objectives[i] for i in range(1, len(objectives))
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(p_values, objectives, 'o-', color='#1B3A5C', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Warehouses (p)', fontsize=12)
ax1.set_ylabel('Total Weighted Distance (million meters)', fontsize=12)
ax1.set_title('p-Median Objective vs. Number of Warehouses', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.4)
ax1.axvline(x=3, color='red', linestyle='--', label='Selected p=3')
ax1.legend()

ax2.bar(p_values, improvements, color='#0D6E6E', alpha=0.8)
ax2.set_xlabel('Number of Warehouses (p)', fontsize=12)
ax2.set_ylabel('Distance Reduction (million meters)', fontsize=12)
ax2.set_title('Marginal Improvement per Added Warehouse', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.4, axis='y')

plt.tight_layout()
plt.savefig('../outputs/facility_coverage_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Elbow curve saved → outputs/facility_coverage_curve.png')

---
## Phase 4 — Multi-Depot Vehicle Routing (CVRP)
Route delivery fleets from **all three** optimal warehouses so every customer is served. Each depot gets its own vehicle fleet with auto-scaled capacity.

> **Concept:** The **Capacitated VRP (CVRP)** partitions customers across vehicles (each with a capacity limit) while minimizing route distance — solved per depot with Google OR-Tools. Routing all depots (rather than one) turns a demo into a complete fleet-planning analysis.

In [ ]:
# ── Step 1: Assign every customer to its nearest optimal warehouse ─────────
# Instead of routing from only one depot, we now build VRP inputs for ALL
# three selected warehouses — so every one of the 150 customers gets routed.
customer_assignment = {}  # warehouse_idx -> list of customer indices
for wh_idx in OPTIMAL_WAREHOUSES:
    customer_assignment[wh_idx] = []

for i in range(n_customers):
    dists_to_open = [haversine_matrix[j][i] for j in OPTIMAL_WAREHOUSES]
    nearest_wh = OPTIMAL_WAREHOUSES[int(np.argmin(dists_to_open))]
    customer_assignment[nearest_wh].append(i)

print('Customer assignment by nearest open warehouse:')
for wh_idx, cust_list in customer_assignment.items():
    demand_sum = sum(customer_demand[i] for i in cust_list)
    print(f'  Warehouse {wh_idx} ({warehouses_gdf.iloc[wh_idx].site_id}): '
          f'{len(cust_list)} customers, {demand_sum} packages')

def build_vrp_inputs(wh_idx, assigned):
    """Build depot-first coordinate lists, demand list, and distance matrix."""
    wh = warehouses_gdf.iloc[wh_idx]
    lats = [wh.geometry.y] + [customer_lats[i] for i in assigned]
    lons = [wh.geometry.x] + [customer_lons[i] for i in assigned]
    demands = [0] + [int(customer_demand[i]) for i in assigned]
    dm = haversine_matrix_vectorized(lats, lons, lats, lons).astype(int)
    np.fill_diagonal(dm, 0)
    return lats, lons, demands, dm

In [ ]:
# ── Step 2: Run OR-Tools CVRP for ALL three warehouses ─────────────────────
import math

def solve_cvrp(dist_matrix, demands, num_vehicles, vehicle_capacity, depot=0,
               time_limit_s=20):
    """Solve CVRP and return per-vehicle routes."""
    manager = pywrapcp.RoutingIndexManager(len(dist_matrix), num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_idx, to_idx):
        f, t = manager.IndexToNode(from_idx), manager.IndexToNode(to_idx)
        return dist_matrix[f][t]

    transit_cb = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_cb)

    def demand_callback(idx):
        return demands[manager.IndexToNode(idx)]

    demand_cb = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_cb, 0, [vehicle_capacity]*num_vehicles, True, 'Capacity'
    )

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    params.time_limit.seconds = time_limit_s

    solution = routing.SolveWithParameters(params)
    if not solution:
        return None, None

    routes, total_distance = [], 0
    for v in range(num_vehicles):
        route, idx = [], routing.Start(v)
        while not routing.IsEnd(idx):
            route.append(manager.IndexToNode(idx))
            idx = solution.Value(routing.NextVar(idx))
        route.append(manager.IndexToNode(idx))
        if len(route) > 2:
            dist = sum(dist_matrix[route[k]][route[k+1]] for k in range(len(route)-1))
            routes.append({'vehicle': v, 'route': route, 'distance_m': dist})
            total_distance += dist
    return routes, total_distance


# ── Solve for each warehouse ───────────────────────────────────────────────
NUM_VEHICLES = 3   # trucks per warehouse (3 warehouses × 3 = 9 total fleet)

depot_solutions = {}   # wh_idx -> dict with routes, distance, inputs
fleet_total_dist = 0

for wh_idx in OPTIMAL_WAREHOUSES:
    assigned = customer_assignment[wh_idx]
    lats, lons, demands, dm = build_vrp_inputs(wh_idx, assigned)

    # Auto-scale capacity: ceil(total demand / vehicles) + 20% buffer
    total_demand = sum(demands)
    cap = max(25, math.ceil(total_demand / NUM_VEHICLES * 1.2))

    print(f'\nWarehouse {wh_idx}: {len(assigned)} customers, '
          f'{total_demand} pkgs, capacity={cap}/truck — solving...')
    routes, dist = solve_cvrp(dm, demands, NUM_VEHICLES, cap)

    if routes is None:  # uncapacitated fallback
        print('  Retrying with relaxed capacity...')
        routes, dist = solve_cvrp(dm, demands, NUM_VEHICLES, total_demand)
    if routes is None:
        raise RuntimeError(f'No CVRP solution for warehouse {wh_idx}. '
                           'Increase NUM_VEHICLES or change the random seed.')

    depot_solutions[wh_idx] = {
        'routes': routes, 'distance_m': dist,
        'lats': lats, 'lons': lons, 'demands': demands, 'dist_matrix': dm,
        'assigned': assigned
    }
    fleet_total_dist += dist
    for r in routes:
        print(f'  Vehicle {r["vehicle"]}: {len(r["route"])-2} stops, '
              f'{r["distance_m"]/1000:.1f} km')

print(f'\n{"="*50}')
print(f'FLEET TOTAL (all 3 warehouses): {fleet_total_dist/1000:.1f} km')
print(f'Total vehicles deployed: '
      f'{sum(len(s["routes"]) for s in depot_solutions.values())}')
print(f'{"="*50}')

In [ ]:
# ── Step 3: Fleet-wide comparison vs. naive nearest-neighbor baseline ──────
def nearest_neighbor_route(dist_matrix, start, unvisited):
    """Greedy nearest-neighbor heuristic route."""
    route, remaining, current = [start], list(unvisited), start
    while remaining:
        nearest = min(remaining, key=lambda j: dist_matrix[current][j])
        route.append(nearest)
        remaining.remove(nearest)
        current = nearest
    route.append(start)
    return route

# Baseline: at each depot, one vehicle serves ALL stops greedily
naive_total_dist = 0
for wh_idx, sol in depot_solutions.items():
    dm = sol['dist_matrix']
    n_stops = len(sol['assigned'])
    nr = nearest_neighbor_route(dm, 0, list(range(1, n_stops+1)))
    nd = sum(dm[nr[k]][nr[k+1]] for k in range(len(nr)-1))
    naive_total_dist += nd

improvement_pct = (naive_total_dist - fleet_total_dist) / naive_total_dist * 100

# Per-vehicle summary table (great for the README)
rows = []
for wh_idx, sol in depot_solutions.items():
    for r in sol['routes']:
        route_demand = sum(sol['demands'][n] for n in r['route'])
        rows.append({
            'warehouse': warehouses_gdf.iloc[wh_idx].site_id,
            'vehicle': f"WH{wh_idx}-V{r['vehicle']}",
            'stops': len(r['route']) - 2,
            'packages': route_demand,
            'distance_km': round(r['distance_m']/1000, 1)
        })
vehicle_summary = pd.DataFrame(rows)

print('='*55)
print('FLEET-WIDE OPTIMIZATION RESULTS')
print('='*55)
print(f'Naive baseline (1 truck/depot):   {naive_total_dist/1000:.1f} km')
print(f'OR-Tools optimized fleet:         {fleet_total_dist/1000:.1f} km')
print(f'Distance reduction:               {improvement_pct:.1f}%')
print(f'Customers served:                 {n_customers} (100%)')
print('='*55)
print('\nPer-vehicle summary:')
display(vehicle_summary)

---
## Phase 5 — Distance Metric Comparison
Quantify how much straight-line (Haversine) distances underestimate real road distances in Chicago.

> **Concept:** The **detour factor** = network_distance / haversine_distance. A value of 1.3 means actual road routes are 30% longer than the straight-line estimate.

In [ ]:
# ── Step 1: Sample 30 depot→customer pairs and compute both distance types ─
# Uses the FIRST optimal warehouse's depot solution from Phase 4.
SAMPLE_SIZE = 30

ref_wh   = OPTIMAL_WAREHOUSES[0]
ref_sol  = depot_solutions[ref_wh]
ref_lats = ref_sol['lats']   # index 0 = depot, 1..n = customers
ref_lons = ref_sol['lons']
n_stops  = len(ref_sol['assigned'])

sample_n   = min(SAMPLE_SIZE, n_stops)
sample_idx = np.random.choice(range(1, n_stops + 1), size=sample_n, replace=False)

haversine_sample = []
network_sample   = []
detour_factors   = []

# Snap depot node once (outside the loop)
orig_node = ox.distance.nearest_nodes(G, X=ref_lons[0], Y=ref_lats[0])

print(f'Computing network distances for {sample_n} sample pairs...')
for i in sample_idx:
    h_dist = haversine_distance(
        ref_lats[0], ref_lons[0],   # Warehouse (depot)
        ref_lats[i], ref_lons[i]    # Customer
    )
    dest_node = ox.distance.nearest_nodes(G, X=ref_lons[i], Y=ref_lats[i])
    try:
        n_dist = nx.shortest_path_length(G, orig_node, dest_node, weight='length')
    except nx.NetworkXNoPath:
        n_dist = np.nan

    if not np.isnan(n_dist) and h_dist > 0:
        haversine_sample.append(h_dist)
        network_sample.append(n_dist)
        detour_factors.append(n_dist / h_dist)

print(f'Computed {len(haversine_sample)} valid distance pairs')
print(f'  Mean Haversine distance: {np.mean(haversine_sample)/1000:.2f} km')
print(f'  Mean Network distance:   {np.mean(network_sample)/1000:.2f} km')
print(f'  Mean Detour Factor:      {np.mean(detour_factors):.3f}')
print(f'  (Network is ~{(np.mean(detour_factors)-1)*100:.0f}% longer than straight-line)')

In [ ]:
# ── Step 2: Plot distance comparison and detour factor distribution ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Haversine vs Network
ax = axes[0]
ax.scatter(np.array(haversine_sample)/1000, np.array(network_sample)/1000,
           alpha=0.6, color='#1B3A5C', s=60)
max_val = max(max(haversine_sample), max(network_sample)) / 1000
ax.plot([0, max_val], [0, max_val], 'r--', label='1:1 line (equal)', linewidth=2)
ax.set_xlabel('Haversine Distance (km)', fontsize=12)
ax.set_ylabel('Network Distance (km)', fontsize=12)
ax.set_title('Haversine vs. Network Distance', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.4)

# Histogram of detour factors
ax = axes[1]
ax.hist(detour_factors, bins=12, color='#0D6E6E', alpha=0.8, edgecolor='white')
ax.axvline(np.mean(detour_factors), color='red', linestyle='--', linewidth=2,
           label=f'Mean = {np.mean(detour_factors):.2f}')
ax.set_xlabel('Detour Factor (Network / Haversine)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Detour Factors', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.4, axis='y')

plt.tight_layout()
plt.savefig('../outputs/distance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Distance comparison plot saved → outputs/distance_comparison.png')

---
## Phase 6 — Interactive Folium Dashboard
Build a layered, interactive HTML map. **Routes are snapped to the road network** — each route leg follows the actual shortest street path, not a straight line.

> **Publishing:** Push `logistics_dashboard.html` to GitHub and enable Pages under Settings → Pages to get a live shareable URL.

In [ ]:
# ── Build the interactive Folium dashboard — routes follow REAL ROADS ──────
# Each route leg is snapped to the network shortest path, so delivery routes
# trace actual Chicago streets instead of straight lines. This is the single
# biggest visual upgrade for the portfolio dashboard.
ROUTE_COLORS = ['#E63946', '#457B9D', '#2D6A4F', '#E9C46A',
                '#F4A261', '#9D4EDD', '#06AED5', '#F72585', '#4361EE']

def route_to_road_geometry(G, lats, lons, route):
    """
    Convert a VRP route (list of stop indices) into lat/lon polyline
    coordinates that follow the road network between consecutive stops.
    Falls back to a straight segment if no network path exists.
    """
    # Snap all stops on this route to their nearest network nodes (batch)
    stop_lons = [lons[i] for i in route]
    stop_lats = [lats[i] for i in route]
    stop_nodes = ox.distance.nearest_nodes(G, X=stop_lons, Y=stop_lats)

    coords = []
    for k in range(len(stop_nodes) - 1):
        try:
            path = nx.shortest_path(G, stop_nodes[k], stop_nodes[k+1],
                                    weight='length')
            coords.extend([(G.nodes[n]['y'], G.nodes[n]['x']) for n in path])
        except nx.NetworkXNoPath:
            coords.append((stop_lats[k],   stop_lons[k]))
            coords.append((stop_lats[k+1], stop_lons[k+1]))
    return coords


m = folium.Map(location=[41.8781, -87.6298], zoom_start=11, tiles='CartoDB positron')

# Layer 1: All customer locations
customer_group = folium.FeatureGroup(name='Customer Stops', show=True)
for i, cu in customers_gdf.iterrows():
    folium.CircleMarker(
        location=[cu.geometry.y, cu.geometry.x],
        radius=4, color='#555555',
        fill=True, fill_color='#AAAAAA', fill_opacity=0.7,
        tooltip=f'Customer {i} | Demand: {customer_demand[i]} pkgs'
    ).add_to(customer_group)
customer_group.add_to(m)

# Layer 2: All candidate warehouse sites (hidden by default)
all_wh_group = folium.FeatureGroup(name='Candidate Warehouses', show=False)
for i, wh in warehouses_gdf.iterrows():
    color = '#C9A84C' if i in OPTIMAL_WAREHOUSES else '#CCCCCC'
    folium.RegularPolygonMarker(
        location=[wh.geometry.y, wh.geometry.x],
        number_of_sides=4, radius=8, rotation=45,
        color=color, fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=f'Warehouse {i} ({wh.site_id})'
    ).add_to(all_wh_group)
all_wh_group.add_to(m)

# Layer 3: Selected optimal warehouses
opt_wh_group = folium.FeatureGroup(name='Optimal Warehouses', show=True)
for j, idx in enumerate(OPTIMAL_WAREHOUSES):
    wh = warehouses_gdf.iloc[idx]
    folium.Marker(
        location=[wh.geometry.y, wh.geometry.x],
        icon=folium.Icon(color='darkblue', icon='home', prefix='fa'),
        tooltip=f'SELECTED Warehouse {idx} ({wh.site_id})',
        popup=folium.Popup(f'<b>Optimal Warehouse {j+1}</b><br>Site: {wh.site_id}',
                           max_width=200)
    ).add_to(opt_wh_group)
opt_wh_group.add_to(m)

# Layer 4: Road-following delivery routes for ALL warehouses
print('Snapping routes to road network (30–90 seconds)...')
vehicle_counter = 0
for wh_idx, sol in depot_solutions.items():
    wh_group = folium.FeatureGroup(
        name=f'Routes — {warehouses_gdf.iloc[wh_idx].site_id}', show=True)
    for r in sol['routes']:
        color = ROUTE_COLORS[vehicle_counter % len(ROUTE_COLORS)]
        vehicle_counter += 1
        road_coords = route_to_road_geometry(
            G, sol['lats'], sol['lons'], r['route'])
        folium.PolyLine(
            road_coords, color=color, weight=3, opacity=0.85,
            tooltip=(f'{warehouses_gdf.iloc[wh_idx].site_id} '
                     f'Vehicle {r["vehicle"]} | '
                     f'{len(r["route"])-2} stops | '
                     f'{r["distance_m"]/1000:.1f} km')
        ).add_to(wh_group)
    wh_group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save('../outputs/logistics_dashboard.html')
print('Dashboard saved → outputs/logistics_dashboard.html')
print('Routes now follow real Chicago streets!')
m  # Display inline

---
## Phase 7 — Results Summary & Portfolio Packaging


In [ ]:
# ── Final Results Summary + Metrics Export ────────────────────────────────
import json as _json

total_vehicles = sum(len(s['routes']) for s in depot_solutions.values())

metrics = {
    'network': {
        'nodes': int(len(nodes_gdf)),
        'edges': int(len(edges_gdf)),
        'total_road_km': round(float(edges_proj['length'].sum()/1000), 0)
    },
    'problem_size': {
        'customers': int(n_customers),
        'candidate_warehouses': int(n_warehouses),
        'total_demand_packages': int(customers_gdf['demand'].sum())
    },
    'p_median': {
        'selected_warehouses': [int(i) for i in OPTIMAL_WAREHOUSES],
        'selected_site_ids': [warehouses_gdf.iloc[i].site_id
                              for i in OPTIMAL_WAREHOUSES]
    },
    'vrp': {
        'naive_baseline_km': round(naive_total_dist/1000, 1),
        'optimized_fleet_km': round(fleet_total_dist/1000, 1),
        'distance_reduction_pct': round(improvement_pct, 1),
        'vehicles_deployed': int(total_vehicles),
        'customers_served_pct': 100.0
    },
    'distance_analysis': {
        'mean_haversine_km': round(float(np.mean(haversine_sample)/1000), 2),
        'mean_network_km': round(float(np.mean(network_sample)/1000), 2),
        'mean_detour_factor': round(float(np.mean(detour_factors)), 3)
    }
}

with open('../outputs/project_metrics.json', 'w') as f:
    _json.dump(metrics, f, indent=2)

vehicle_summary.to_csv('../outputs/vehicle_summary.csv', index=False)

print('='*60)
print('PROJECT 01 — FINAL RESULTS SUMMARY')
print('='*60)
print(f'Road network nodes:            {len(nodes_gdf):,}')
print(f'Road network edges:            {len(edges_gdf):,}')
print(f'Customer demand points:        {n_customers} (100% routed)')
print(f'Optimal warehouses (p=3):      {OPTIMAL_WAREHOUSES}')
print()
print(f'Naive baseline distance:       {naive_total_dist/1000:.1f} km')
print(f'Optimized fleet distance:      {fleet_total_dist/1000:.1f} km')
print(f'Distance reduction:            {improvement_pct:.1f}%')
print(f'Vehicles deployed:             {total_vehicles} across 3 depots')
print()
print(f'Mean detour factor:            {np.mean(detour_factors):.3f}')
print('='*60)
print()
print('Output files generated:')
print('  - chicago_network.png')
print('  - facility_coverage_curve.png')
print('  - distance_comparison.png')
print('  - logistics_dashboard.html   (road-following routes)')
print('  - project_metrics.json       (README metrics)')
print('  - vehicle_summary.csv        (per-vehicle table)')

---
## Summary

This notebook solves a 3-warehouse p-median facility location problem and a multi-depot capacitated VRP on Chicago's real road network, then compares straight-line vs. network distance assumptions. See the project `README.md` for the results table, repo structure, and setup instructions (`SETUP.md`).
